# Modulo 3 · Lezione 4 — Bash scripting

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | Struttura di uno script Bash | Shebang, commenti, `chmod +x` |
| 2 | Variabili | Dichiarare, usare, variabili speciali |
| 3 | Argomenti | `$1`, `$2`, `$@`, `$#` |
| 4 | Condizionali | `if`, `elif`, `else`, operatori di confronto |
| 5 | Cicli | `for`, `while` |
| 6 | Funzioni | Dichiarare e chiamare funzioni |
| 7 | Pratica | Script di automazione per progetti IA |

> 💡 **Idea centrale:** Uno script Bash è il modo più diretto per automatizzare compiti ripetitivi sul server. Nell'IA lo userai per lanciare esperimenti, elaborare dati, fare backup automatici e preparare ambienti.

---
## 1. Struttura di uno script Bash

```bash
#!/bin/bash
# Il mio primo script — commento descrittivo

echo "Ciao da Bash"
```

**Riga 1: lo shebang `#!/bin/bash`**
Dice al sistema quale interprete usare. Senza di esso, il sistema cercherà di indovinare e potrebbe fallire.

```bash
#!/bin/bash        # Bash (il più comune)
#!/usr/bin/env bash # Cerca bash nel PATH (più portabile)
#!/bin/sh          # Shell POSIX di base (compatibile su più sistemi)
#!/usr/bin/env python3  # Funziona anche con Python
```

**Per eseguire uno script:**
```bash
# Opzione 1: renderlo eseguibile e lanciarlo
chmod +x mi_script.sh
./mi_script.sh

# Opzione 2: passarlo direttamente a bash
bash mi_script.sh
```

**Buone pratiche fin dall'inizio:**
```bash
#!/usr/bin/env bash
set -e    # esce se un comando fallisce
set -u    # errore se si usa una variabile non definita
set -o pipefail  # rileva errori nelle pipe
```

In [ ]:
from pathlib import Path
import os, subprocess

SCRIPTS = Path.home() / "bash_scripts"
SCRIPTS.mkdir(exist_ok=True)
os.chdir(SCRIPTS)

# Scrivere il primo script
Path("hola.sh").write_text("""\
#!/usr/bin/env bash
# Primo script: saluto di base

echo "Ciao da Bash"
echo "Data: $(date +%Y-%m-%d)"
echo "Utente: $USER"
echo "Directory: $(pwd)"
""", encoding="utf-8")

!chmod +x hola.sh
print("Script creato ed eseguibile. Esecuzione:")
!bash hola.sh

---
## 2. Variabili in Bash

```bash
# Dichiarare (senza spazi attorno all'=)
NOMBRE="Ana"
NUMERO=42
LISTA="a b c"

# Usare (con il prefisso $)
echo "Ciao, $NOMBRE"
echo "Il numero è: ${NUMERO}"
echo "Testo: ${NOMBRE}_suffisso"   # graffe per delimitare

# Sostituzione di comando (il risultato diventa il valore)
FECHA=$(date +%Y-%m-%d)
LINEAS=$(wc -l < archivo.txt)

# Variabili d'ambiente (disponibili globalmente)
export MI_VAR="valore"            # disponibile per i sottoprocessi
```

**Variabili speciali di sistema:**

| Variabile | Significato |
|----------|-------------|
| `$HOME`  | Directory personale dell'utente |
| `$USER`  | Nome dell'utente |
| `$PATH`  | Directory in cui si cercano gli eseguibili |
| `$PWD`   | Directory corrente |
| `$?`     | Codice di uscita dell'ultimo comando (0 = successo) |
| `$$`     | PID del processo corrente |

In [ ]:
Path("variables.sh").write_text("""\
#!/usr/bin/env bash

# Dichiarare variabili
PROYECTO="sistema_biblioteca"
VERSION="1.0.0"
FECHA=$(date +%Y-%m-%d)

# Usare le variabili
echo "Progetto: $PROYECTO"
echo "Versione: $VERSION"
echo "Data:     $FECHA"

# Variabili di sistema
echo "Utente:   $USER"
echo "Home:     $HOME"
echo "Pwd:      $PWD"

# Operazioni aritmetiche (con doppie parentesi)
A=10
B=3
echo "$A + $B = $((A + B))"
echo "$A * $B = $((A * B))"
echo "$A / $B = $((A / B))"

# Codice di uscita
ls /tmp > /dev/null 2>&1
echo "ls /tmp ha funzionato: $?"

ls /ruta_inexistente > /dev/null 2>&1
echo "ls ruta_inexistente è fallito: $?"
""", encoding="utf-8")

!bash variables.sh

---
## 3. Argomenti

Gli argomenti che passi allo script quando lo esegui sono disponibili come variabili posizionali:

```bash
bash mi_script.sh arg1 arg2 arg3

# All'interno dello script:
$0    # nome dello script
$1    # primo argomento
$2    # secondo argomento
$@    # tutti gli argomenti (come lista)
$*    # tutti gli argomenti (come stringa)
$#    # numero di argomenti
```

In [ ]:
Path("con_argumentos.sh").write_text("""\
#!/usr/bin/env bash
# Uso: bash con_argumentos.sh <nome> <versione>

NOMBRE=${1:-"proyecto"}     # valore di default se non viene passato
VERSION=${2:-"0.0.1"}

echo "Script: $0"
echo "Nome: $NOMBRE"
echo "Versione: $VERSION"
echo "Argomenti ricevuti: $#"
echo "Tutti: $@"

# Iterare su tutti gli argomenti
echo "\nScorrendo gli argomenti:"
for arg in "$@"; do
    echo "  - $arg"
done
""", encoding="utf-8")

print("=== Senza argomenti (valori di default) ===")
!bash con_argumentos.sh

print("\n=== Con argomenti ===")
!bash con_argumentos.sh mi_proyecto 2.1.0 extra

---
## 4. Condizionali

```bash
if [ condizione ]; then
    # comandi se vero
elif [ altra_condizione ]; then
    # comandi se elif vero
else
    # comandi se tutto falso
fi
```

**Confronto di stringhe:**

| Operatore | Significato |
|----------|-----------|
| `"$a" = "$b"` | uguale |
| `"$a" != "$b"` | diverso |
| `-z "$a"` | stringa vuota |
| `-n "$a"` | stringa non vuota |

**Confronto numerico:**

| Operatore | Significato |
|----------|-----------|
| `-eq` | uguale |
| `-ne` | diverso |
| `-lt` | minore di |
| `-le` | minore o uguale |
| `-gt` | maggiore di |
| `-ge` | maggiore o uguale |

**File e directory:**

| Operatore | Significato |
|----------|-----------|
| `-f file` | esiste ed è un file |
| `-d dir` | esiste ed è una directory |
| `-e percorso` | esiste (qualsiasi tipo) |
| `-r file` | ha permessi di lettura |
| `-x file` | è eseguibile |

In [ ]:
Path("condicionales.sh").write_text("""\
#!/usr/bin/env bash

ARCHIVO=$1

# Verificare l'argomento
if [ -z "$ARCHIVO" ]; then
    echo "Errore: devi passare un file come argomento"
    exit 1
fi

# Verificare se esiste
if [ -e "$ARCHIVO" ]; then
    if [ -f "$ARCHIVO" ]; then
        LINEAS=$(wc -l < "$ARCHIVO")
        echo "File trovato: $ARCHIVO"
        echo "Righe: $LINEAS"

        if [ $LINEAS -gt 10 ]; then
            echo "È un file grande (più di 10 righe)"
        else
            echo "È un file piccolo (10 righe o meno)"
        fi

    elif [ -d "$ARCHIVO" ]; then
        echo "È una directory, non un file"
    fi
else
    echo "Non esiste: $ARCHIVO"
    exit 1
fi
""", encoding="utf-8")

print("=== Senza argomento ===")
!bash condicionales.sh

print("\n=== File esistente ===")
!bash condicionales.sh variables.sh

print("\n=== File inesistente ===")
!bash condicionales.sh no_existe.txt

---
## 5. Cicli

### `for` — iterare su una lista

```bash
# Iterare su una lista letterale
for item in a b c d; do
    echo "$item"
done

# Iterare sui file (glob)
for archivo in *.py; do
    echo "Elaborazione: $archivo"
done

# Iterare su un intervallo numerico
for i in $(seq 1 10); do
    echo "Iterazione $i"
done

# Stile C
for ((i=0; i<5; i++)); do
    echo $i
done
```

### `while` — iterare finché la condizione è vera

```bash
CONTADOR=0
while [ $CONTADOR -lt 5 ]; do
    echo "Contatore: $CONTADOR"
    ((CONTADOR++))
done

# Leggere un file riga per riga
while IFS= read -r linea; do
    echo "Riga: $linea"
done < archivo.txt
```

In [ ]:
Path("bucles.sh").write_text("""\
#!/usr/bin/env bash

echo "=== for su lista ==="
for extension in py sh txt csv log; do
    COUNT=$(find . -name "*.$extension" | wc -l)
    echo "  .$extension: $COUNT file"
done

echo ""
echo "=== for su glob ==="
for script in *.sh; do
    LINEAS=$(wc -l < "$script")
    echo "  $script: $LINEAS righe"
done

echo ""
echo "=== for numerico (simulazione di 5 epoche) ==="
for epoch in $(seq 1 5); do
    echo "  Epoch $epoch/5 — elaborazione..."
done

echo ""
echo "=== while leggendo un file ==="
CONTADOR=0
while IFS= read -r linea; do
    ((CONTADOR++))
done < variables.sh
echo "  variables.sh ha $CONTADOR righe"
""", encoding="utf-8")

!bash bucles.sh

---
## 6. Funzioni

```bash
# Dichiarare una funzione
nombre_funcion() {
    local variable_local="valore"   # local: visibile solo nella funzione
    echo "$variable_local"
    echo "Argomento 1: $1"
    return 0  # codice di uscita (0 = successo)
}

# Chiamare la funzione
nombre_funcion "argomento"

# Catturare l'output di una funzione
resultado=$(nombre_funcion "argomento")
echo "Risultato: $resultado"
```

**Punti importanti:**
- Le funzioni vanno dichiarate **prima** di usarle.
- `local` evita di contaminare lo scope globale.
- `return` restituisce un codice di uscita (0-255), non un valore. Per restituire dei valori, usa `echo` e cattura con `$(...)`.

In [ ]:
Path("funciones.sh").write_text("""\
#!/usr/bin/env bash

# Funzione: stampare un separatore con titolo
titulo() {
    local texto="$1"
    local largo=${#texto}
    printf '=%.0s' $(seq 1 $((largo + 4)))
    echo ""
    echo "  $texto"
    printf '=%.0s' $(seq 1 $((largo + 4)))
    echo ""
}

# Funzione: verificare se una directory esiste, crearla se no
asegurar_directorio() {
    local dir="$1"
    if [ ! -d "$dir" ]; then
        mkdir -p "$dir"
        echo "  Creata: $dir"
    else
        echo "  Esiste già: $dir"
    fi
}

# Funzione: contare i file per estensione
contar_archivos() {
    local directorio="$1"
    local extension="$2"
    local count
    count=$(find "$directorio" -name "*.$extension" -type f 2>/dev/null | wc -l)
    echo $count
}

# --- Programma principale ---
titulo "Demo delle Funzioni"

echo ""
titulo "Verificare le Directory"
asegurar_directorio "/tmp/test_bash_dir"
asegurar_directorio "/tmp/test_bash_dir"

echo ""
titulo "Contare gli Script"
N=$(contar_archivos "." "sh")
echo "  Script .sh nella directory corrente: $N"
""", encoding="utf-8")

!bash funciones.sh

---
## 7. Pratica — Script di automazione per progetti IA

---

### Esercizio 1 — Script di verifica dell'ambiente

**Consegna:** Scrivi `verificar_entorno.sh` che verifica che l'ambiente di sviluppo sia configurato correttamente:

1. Verifica che Python 3 sia installato e mostra la sua versione.
2. Verifica che Git sia installato.
3. Verifica che la directory `~/ml_proyecto` esista.
4. Verifica che esista il file `requirements.txt` al suo interno.
5. Mostra un riepilogo: quante verifiche sono passate e quante sono fallite.
6. Esce con codice 1 se qualche verifica è fallita.

In [ ]:
# ✏️ IL TUO CODICE QUI
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```bash
#!/usr/bin/env bash
# verificar_entorno.sh — verifica l'ambiente di sviluppo

PASARON=0
FALLARON=0

check() {
    local descripcion="$1"
    local resultado="$2"   # 0 = ok, 1 = fallito
    local detalle="$3"

    if [ "$resultado" -eq 0 ]; then
        echo "  ✅ $descripcion${detalle:+ — $detalle}"
        ((PASARON++))
    else
        echo "  ❌ $descripcion"
        ((FALLARON++))
    fi
}

echo "============================="
echo "  VERIFICA DELL'AMBIENTE"
echo "============================="
echo ""

# 1. Python 3
if command -v python3 &> /dev/null; then
    VERSION=$(python3 --version 2>&1)
    check "Python 3 installato" 0 "$VERSION"
else
    check "Python 3 installato" 1
fi

# 2. Git
if command -v git &> /dev/null; then
    GIT_V=$(git --version)
    check "Git installato" 0 "$GIT_V"
else
    check "Git installato" 1
fi

# 3. Directory ml_proyecto
if [ -d "$HOME/ml_proyecto" ]; then
    check "Directory ~/ml_proyecto" 0
else
    check "Directory ~/ml_proyecto" 1
fi

# 4. requirements.txt
if [ -f "$HOME/ml_proyecto/requirements.txt" ]; then
    check "requirements.txt esiste" 0
else
    check "requirements.txt esiste" 1
fi

# Riepilogo
echo ""
echo "Verifiche: $PASARON passate, $FALLARON fallite"

if [ $FALLARON -gt 0 ]; then
    echo "⚠️  Ci sono verifiche fallite"
    exit 1
else
    echo "✅ Ambiente OK"
    exit 0
fi
```

**`command -v nombre`** è il modo corretto per verificare se un eseguibile esiste nel PATH (più portabile di `which`).
</details>

---
### Esercizio 2 — Script di lancio degli esperimenti

**Consegna:** Scrivi `lanzar_experimento.sh` che simula il lancio di più esperimenti di ML:

```bash
bash lanzar_experimento.sh <nome_progetto> <lr1> <lr2> <lr3>
# Esempio: bash lanzar_experimento.sh mi_modelo 0.001 0.01 0.1
```

Lo script deve:
1. Verificare che siano stati passati almeno due argomenti.
2. Creare la cartella `experimentos/<nome_progetto>/<data>/`.
3. Per ogni learning rate, creare un file `config_lr_<lr>.yaml` con il contenuto: `learning_rate: <lr>` e un timestamp.
4. Simulare l'addestramento: per ogni config, stampare `Lanzando experimento con lr=<lr>...` e creare un log vuoto.
5. Alla fine, elencare tutti i file creati.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```bash
#!/usr/bin/env bash
# lanzar_experimento.sh — simula il lancio di esperimenti con diversi LR

# 1. Verificare gli argomenti
if [ $# -lt 2 ]; then
    echo "Uso: $0 <nome_progetto> <lr1> [lr2] [lr3...]"
    echo "Esempio: $0 red_v1 0.001 0.01 0.1"
    exit 1
fi

PROYECTO=$1
shift             # elimina $1, i LR restano in $@
FECHA=$(date +%Y%m%d_%H%M%S)
BASE="experimentos/$PROYECTO/$FECHA"

echo "======================================"
echo "  ESPERIMENTO: $PROYECTO"
echo "  Data: $FECHA"
echo "  Learning rate: $@"
echo "======================================"

# 2. Creare la directory base
mkdir -p "$BASE"
echo "Directory creata: $BASE"

# 3 e 4. Per ogni learning rate
for lr in "$@"; do
    CONFIG="$BASE/config_lr_${lr}.yaml"
    LOG="$BASE/train_lr_${lr}.log"

    # Creare la config
    cat > "$CONFIG" << EOF
# Config generata automaticamente
proyecto: $PROYECTO
fecha: $FECHA
learning_rate: $lr
epochs: 50
batch_size: 32
EOF

    echo "Lanzando experimento con lr=$lr..."
    touch "$LOG"    # in un caso reale: python train.py --config $CONFIG &> $LOG
    echo "  Config: $CONFIG"
    echo "  Log:    $LOG"
done

# 5. Elencare i file creati
echo ""
echo "File creati:"
find "$BASE" -type f
echo ""
echo "✅ $(find "$BASE" -type f | wc -l) file generati"
```

**`shift`** sposta gli argomenti: dopo `shift`, `$1` diventa il precedente `$2`, e i LR rimasti in `$@` si possono scorrere con `for`.

**`cat << EOF ... EOF`** è un *heredoc* — un modo comodo per scrivere testo multilinea in un file.
</details>

---
### Esercizio 3 — Script di pulizia del progetto

**Consegna:** Scrivi `limpiar_proyecto.sh <directory>` che elimina i file temporanei e le cache di un progetto Python:

1. Verifica che la directory esista e sia accessibile.
2. Cerca ed elimina: `__pycache__/`, `*.pyc`, `*.pyo`, `*.tmp`, `*.log` (con `-delete`).
3. Riporta quanti file/cartelle ha eliminato per ogni tipo.
4. Mostra lo spazio totale recuperato prima e dopo (usa `du -sh`).
5. Chiede conferma prima di eliminare (mostra cosa verrà eliminato e attende `s/n`).

In [ ]:
# ✏️ IL TUO CODICE QUI


---
### Esercizio 4 — Script di monitoraggio dei log

**Consegna:** Scrivi `monitorear_logs.sh <directory_log>` che genera un report sullo stato dei log:

1. Elenca tutti i `.log` ordinati per data di modifica (più recente prima).
2. Per ogni file, mostra: nome, dimensione, numero di righe, numero di errori.
3. Calcola il totale degli errori in tutti i log.
4. Salva il report in `resumen_logs_<data>.txt`.
5. Se ci sono più di 10 errori in totale, aggiunge un avviso in fondo al report.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```bash
#!/usr/bin/env bash
# monitorear_logs.sh — report dei log

DIR=${1:-"."}
FECHA=$(date +%Y%m%d_%H%M%S)
INFORME="resumen_logs_${FECHA}.txt"

if [ ! -d "$DIR" ]; then
    echo "Errore: '$DIR' non è una directory" >&2
    exit 1
fi

TOTAL_ERRORES=0

# Intestazione
{
echo "REPORT DEI LOG — $(date)"
echo "Directory: $DIR"
echo ""
printf '%-30s %8s %8s %8s\n' "FILE" "DIMENS." "RIGHE" "ERRORI"
printf '%s\n' "$(printf '─%.0s' {1..58})"

# 1 e 2. Log ordinati per data
while IFS= read -r log; do
    nombre=$(basename "$log")
    tamano=$(du -sh "$log" | cut -f1)
    lineas=$(wc -l < "$log")
    errores=$(grep -c 'ERROR' "$log" 2>/dev/null || echo 0)

    printf '%-30s %8s %8s %8s\n' "$nombre" "$tamano" "$lineas" "$errores"
    TOTAL_ERRORES=$((TOTAL_ERRORES + errores))
done < <(find "$DIR" -name "*.log" -type f -printf '%T@ %p\n' | sort -rn | cut -d' ' -f2-)

# 3. Totale
echo ""
echo "Totale errori: $TOTAL_ERRORES"

# 5. Avviso
if [ $TOTAL_ERRORES -gt 10 ]; then
    echo ""
    echo "⚠️  AVVISO: trovati $TOTAL_ERRORES errori (soglia: 10)"
    echo "   Controlla i file di log per identificare il problema."
fi

} | tee "$INFORME"

echo ""
echo "Report salvato in: $INFORME"
```

**`{ ... } | tee file`** invia il blocco di comandi sia allo schermo sia al file contemporaneamente. **`< <(...)`** è *process substitution* — un modo avanzato per alimentare un `while` con l'output di un altro comando.
</details>

---
### Esercizio 5 (sfida) — Script di backup con rotazione

**Consegna:** Scrivi `backup_rotacion.sh <origine> <destinazione> <max_backup>` che implementa un sistema di backup con rotazione automatica:

1. Crea un backup della directory `origine` in `destinazione/backup_YYYYMMDD_HHMMSS.tar.gz`.
2. Dopo aver creato il backup, conta quanti ce ne sono in `destinazione/`.
3. Se ce ne sono più di `max_backup`, elimina i più vecchi (mantiene solo i `max_backup` più recenti).
4. Verifica che il backup sia stato creato correttamente (che il file esista e abbia dimensione > 0).
5. Genera un log dell'operazione in `destinazione/backup.log` con: data, dimensione del backup, backup eliminati.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```bash
#!/usr/bin/env bash
set -euo pipefail
# backup_rotacion.sh — backup con rotazione automatica

ORIGEN=${1:?"Argomento 1 richiesto: directory di origine"}
DESTINO=${2:?"Argomento 2 richiesto: directory di destinazione"}
MAX=${3:-5}  # per default, conserva 5 backup

FECHA=$(date +%Y%m%d_%H%M%S)
NOMBRE_BACKUP="backup_${FECHA}.tar.gz"
RUTA_BACKUP="$DESTINO/$NOMBRE_BACKUP"
LOG="$DESTINO/backup.log"

mkdir -p "$DESTINO"

log_msg() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] $*" | tee -a "$LOG"
}

log_msg "=== Inizio backup ==="
log_msg "Origine:      $ORIGEN"
log_msg "Destinazione: $RUTA_BACKUP"

# 1. Creare il backup
if tar -czf "$RUTA_BACKUP" -C "$(dirname "$ORIGEN")" "$(basename "$ORIGEN")" 2>/dev/null; then
    TAMANO=$(du -sh "$RUTA_BACKUP" | cut -f1)
    log_msg "Backup creato: $TAMANO"
else
    log_msg "ERRORE: creazione del backup fallita"
    exit 1
fi

# 4. Verificare che esista e abbia dimensione
if [ ! -s "$RUTA_BACKUP" ]; then
    log_msg "ERRORE: il backup è vuoto"
    exit 1
fi
log_msg "Verifica OK"

# 2 e 3. Rotazione: eliminare i backup in eccesso
TOTAL=$(find "$DESTINO" -maxdepth 1 -name "backup_*.tar.gz" | wc -l)
log_msg "Backup esistenti: $TOTAL (massimo: $MAX)"

ELIMINADOS=0
while [ "$TOTAL" -gt "$MAX" ]; do
    MAS_ANTIGUO=$(find "$DESTINO" -maxdepth 1 -name "backup_*.tar.gz" | sort | head -1)
    log_msg "Eliminazione: $(basename "$MAS_ANTIGUO")"
    rm "$MAS_ANTIGUO"
    ((TOTAL--))
    ((ELIMINADOS++))
done

log_msg "Backup eliminati: $ELIMINADOS"
log_msg "Backup attuali: $(find "$DESTINO" -maxdepth 1 -name 'backup_*.tar.gz' | wc -l)"
log_msg "=== Fine backup OK ==="
echo ""
echo "✅ Backup completato: $RUTA_BACKUP ($TAMANO)"
```

**`${VAR:?"messaggio"}`** fallisce con il messaggio indicato se la variabile non è definita — meglio che verificare manualmente con `if`.

**`-s`** nei condizionali sui file verifica che il file esista E abbia dimensione maggiore di zero.
</details>

---
## Riepilogo della lezione

```bash
# ─── Struttura ─────────────────────────────────────────────
#!/usr/bin/env bash          # shebang
set -euo pipefail            # esce in caso di errore, variabili non definite
chmod +x script.sh           # rendere eseguibile
bash script.sh               # eseguire

# ─── Variabili ─────────────────────────────────────────────
VAR="valore"                  # dichiarare (senza spazi)
echo "$VAR"                  # usare
RESULTADO=$(comando)         # sostituzione di comando
$((A + B))                   # aritmetica
${VAR:-"default"}           # valore di default

# ─── Argomenti ─────────────────────────────────────────────
$1 $2 ...                    # argomenti posizionali
$@ / $*                      # tutti gli argomenti
$#                           # numero di argomenti
shift                        # spostare gli argomenti

# ─── Condizionali ──────────────────────────────────────────
if [ cond ]; then ... fi     # base
[ -f file ]                  # è un file
[ -d dir ]                   # è una directory
[ -z "$VAR" ]                # vuota
[ $A -gt $B ]                # maggiore di
command -v prog              # è installato

# ─── Cicli ─────────────────────────────────────────────────
for item in lista; do ... done
for f in *.py; do ... done   # sui file
for i in $(seq 1 10); do ... done
while [ cond ]; do ... done
while IFS= read -r line; do ... done < archivo

# ─── Funzioni ──────────────────────────────────────────────
nombre() { local v="..."; echo "$v"; }
resultado=$(nombre)
```

## Cosa vedremo nella prossima lezione?

La **Lezione 5** è il progetto integrativo: scriviamo uno script completo di automazione che combina tutto il Modulo 3 — gestione dei file, ricerca, redirezioni e Bash scripting.